# 🔄 Flujos de trabajo básicos de agentes con Microsoft Foundry (Python)

## 📋 Tutorial de orquestación de flujos de trabajo

Este cuaderno introduce las potentes capacidades del **Constructor de Flujos de Trabajo** del Microsoft Agent Framework. Aprende a crear flujos de trabajo de agentes sofisticados y con múltiples pasos que pueden manejar procesos empresariales complejos y coordinar múltiples operaciones de IA sin problemas.

> **Nota de migración:** Este ejemplo hacía referencia anteriormente a Modelos de GitHub. Modelos de GitHub está en desuso (se retirará en julio de 2026), por lo que ahora utiliza **Microsoft Foundry** a través del `FoundryChatClient`, que apunta a la **API de Respuestas** de Azure OpenAI.

## 🎯 Objetivos de aprendizaje

### 🏗️ **Arquitectura del flujo de trabajo**
- **Constructor de flujos de trabajo**: Diseñar y orquestar procesos complejos de varios pasos
- **Ejecución basada en eventos**: Manejar eventos y transiciones de estado del flujo de trabajo
- **Diseño visual del flujo de trabajo**: Crear y visualizar estructuras de flujos de trabajo
- **Integración con Microsoft Foundry**: Aprovechar modelos de IA dentro de contextos de flujo de trabajo

### 🔄 **Orquestación de procesos**
- **Operaciones secuenciales**: Encadenar múltiples tareas de agentes en orden lógico
- **Lógica condicional**: Implementar puntos de decisión y ramificaciones en los flujos de trabajo
- **Manejo de errores**: Recuperación robusta y resiliencia del flujo de trabajo
- **Gestión de estado**: Rastrear y administrar el estado de ejecución del flujo de trabajo

### 📊 **Patrones de flujo de trabajo empresarial**
- **Automatización de procesos empresariales**: Automatizar flujos de trabajo organizacionales complejos
- **Coordinación multiagente**: Coordinar múltiples agentes especializados
- **Ejecución escalable**: Diseñar flujos de trabajo para operaciones a escala empresarial
- **Monitoreo y observabilidad**: Monitorear el rendimiento y los resultados del flujo de trabajo

## ⚙️ Requisitos previos y configuración

### 📦 **Dependencias requeridas**

Instala el Agent Framework con capacidades de flujo de trabajo:

```bash
pip install agent-framework -U
```

### 🔑 **Configuración de Microsoft Foundry**

Inicia sesión con Azure CLI (`az login`) para que `AzureCliCredential` pueda autenticarse, luego configura los detalles de tu proyecto Microsoft Foundry.

**Configuración del entorno (archivo .env):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

### 🏢 **Casos de uso empresarial**

**Ejemplos de procesos empresariales:**
- **Incorporación de clientes**: Flujos de trabajo de verificación y configuración en múltiples pasos
- **Canal de contenido**: Creación, revisión y publicación automatizada de contenido
- **Procesamiento de datos**: Flujos ETL con transformación impulsada por IA
- **Aseguramiento de calidad**: Procesos automatizados de prueba y validación

**Beneficios del flujo de trabajo:**
- 🎯 **Confiabilidad**: Ejecución determinista con recuperación de errores
- 📈 **Escalabilidad**: Manejar automatización de procesos de alto volumen
- 🔍 **Observabilidad**: Rastreos completos de auditoría y monitoreo
- 🔧 **Mantenibilidad**: Diseño visual y componentes modulares

## 🎨 Patrones de diseño de flujos de trabajo

### Estructura básica del flujo de trabajo
```mermaid
graph TD
    A[Inicio] --> B[Tarea del Agente 1]
    B --> C{Punto de Decisión}
    C -->|Éxito| D[Tarea del Agente 2]
    C -->|Fallo| E[Manejador de Errores]
    D --> F[Fin]
    E --> F
```

**Componentes clave:**
- **WorkflowBuilder**: Motor principal de orquestación
- **WorkflowEvent**: Manejo de eventos y comunicación
- **WorkflowViz**: Representación visual y depuración del flujo de trabajo

¡Construyamos tu primer flujo de trabajo inteligente! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
